In [1]:
import requests
import json
import time
import pandas as pd



In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
URL = "https://xeroprod.org.coveo.com/rest/search/v2?organizationId=xeroprod"
TOKEN = "Bearer xx3ca09408-dde7-4bdc-936e-6e7e722c4025"
BATCH_SIZE = 50
INDUSTRY = "Construction and trades"
LOCATION = "United Kingdom"

HEADERS = {
    "Authorization": TOKEN,
    "Content-Type": "application/json",
    "Accept": "*/*",
    "Origin": "https://www.xero.com",
    "Referer": "https://www.xero.com/",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}


In [5]:



def build_payload(first_result):
    return {
        "locale": "en-US",
        "debug": True,
        "tab": "default",
        "referrer": "default",
        "timezone": "Europe/London",
        "pipeline": "Advisor Directory",
        "searchHub": "Advisor directory",
        "q": LOCATION,
        "enableQuerySyntax": False,
        "numberOfResults": BATCH_SIZE,
        "firstResult": first_result,
        "fieldsToInclude": [
            "author", "language", "urihash", "objecttype", "collection",
            "source", "permanentid", "date", "filetype", "parents",
            "ec_price", "ec_name", "ec_description", "ec_brand", "ec_category",
            "ec_item_group_id", "ec_shortdesc", "ec_thumbnails", "ec_images",
            "ec_promo_price", "ec_in_stock", "ec_rating", "adlistingid",
            "imagesrc", "description"
        ],
        "facets": [
            {
                "filterFacetCount": True,
                "injectionDepth": 1000,
                "numberOfValues": 10,
                "sortCriteria": "alphanumeric",
                "resultsMustMatch": "atLeastOneValue",
                "type": "specific",
                "currentValues": [
                    {"value": INDUSTRY, "state": "selected"}
                ],
                "freezeCurrentValues": True,
                "isFieldExpanded": False,
                "preventAutoSelect": True,
                "facetId": "adindustry",
                "field": "adindustry"
            },
            {
                "delimitingCharacter": "|",
                "filterFacetCount": True,
                "injectionDepth": 1000,
                "numberOfValues": 1,
                "sortCriteria": "occurrences",
                "basePath": [],
                "filterByBasePath": True,
                "resultsMustMatch": "atLeastOneValue",
                "currentValues": [
                    {
                        "value": LOCATION,
                        "state": "selected",
                        "children": [],
                        "retrieveChildren": True,
                        "retrieveCount": 10
                    }
                ],
                "preventAutoSelect": False,
                "type": "hierarchical",
                "facetId": "adcountryregionhierarchy",
                "field": "adcountryregionhierarchy"
            }
        ],
        "facetOptions": {"freezeFacetOrder": True}
    }

def extract_fields(result):
    raw = result.get("raw", {})
    return {
        "title":        result.get("title", ""),
        "uri":          result.get("uri", ""),
        "description":  result.get("excerpt", "") or raw.get("description", ""),
        "listing_id":   raw.get("adlistingid", ""),
        "industry":     raw.get("adindustry", ""),
        "location":     raw.get("adcountryregionhierarchy", ""),
        "source":       raw.get("source", ""),
        "date":         raw.get("date", ""),
        "image":        raw.get("imagesrc", ""),
        "permanent_id": raw.get("permanentid", ""),
    }


In [7]:

# ── STEP 1: Run this first to preview fields ──────────────────────────────────
# print("Making first request...")
# r = requests.post(URL, headers=HEADERS, json=build_payload(0))
# print(f"Status: {r.status_code}")

# if r.status_code != 200:
#     print("Error:", r.text[:500])
#     exit()

# data = r.json()
# total = data.get("totalCount", data.get("totalCountFiltered", 0))
# print(f"\nTotal results for '{INDUSTRY}' in '{LOCATION}': {total}")

# if data.get("results"):
#     print("\n── RAW FIRST RESULT (all available fields) ──────────────────────")
#     print(json.dumps(data["results"][0], indent=2))
#     print("\n── EXTRACTED FIELDS ─────────────────────────────────────────────")
#     print(json.dumps(extract_fields(data["results"][0]), indent=2))

# ── STEP 2: Once happy with fields, uncomment to scrape everything ────────────

all_results = []
batch = 0
while True:
    first = batch * BATCH_SIZE
    if first >= total:
        break
    print(f"Fetching {first}–{first + BATCH_SIZE} of {total}...")
    r = requests.post(URL, headers=HEADERS, json=build_payload(first))
    if r.status_code != 200:
        print(f"Error on batch {batch}: {r.status_code} — {r.text[:200]}")
        break
    results = r.json().get("results", [])
    if not results:
        break
    all_results.extend([extract_fields(res) for res in results])
    batch += 1
    time.sleep(0.5)

print(f"\nTotal scraped: {len(all_results)}")
df = pd.DataFrame(all_results)
df.to_csv("xero_construction_advisors.csv", index=False)
df.to_excel("xero_construction_advisors.xlsx", index=False)
print("Saved: xero_construction_advisors.csv + .xlsx")

Fetching 0–50 of 2926...
Fetching 50–100 of 2926...
Fetching 100–150 of 2926...
Fetching 150–200 of 2926...
Fetching 200–250 of 2926...
Fetching 250–300 of 2926...
Fetching 300–350 of 2926...
Fetching 350–400 of 2926...
Fetching 400–450 of 2926...
Fetching 450–500 of 2926...
Fetching 500–550 of 2926...
Fetching 550–600 of 2926...
Fetching 600–650 of 2926...
Fetching 650–700 of 2926...
Fetching 700–750 of 2926...
Fetching 750–800 of 2926...
Fetching 800–850 of 2926...
Fetching 850–900 of 2926...
Fetching 900–950 of 2926...
Fetching 950–1000 of 2926...
Fetching 1000–1050 of 2926...
Fetching 1050–1100 of 2926...
Fetching 1100–1150 of 2926...
Fetching 1150–1200 of 2926...
Fetching 1200–1250 of 2926...
Fetching 1250–1300 of 2926...
Fetching 1300–1350 of 2926...
Fetching 1350–1400 of 2926...
Fetching 1400–1450 of 2926...
Fetching 1450–1500 of 2926...
Fetching 1500–1550 of 2926...
Fetching 1550–1600 of 2926...
Fetching 1600–1650 of 2926...
Fetching 1650–1700 of 2926...
Fetching 1700–1750 of 2

ModuleNotFoundError: No module named 'openpyxl'